# UAS BENGKEL KODING DATA SCIENCE
## Prediksi Churn Pelanggan – Sales and Marketing Dataset

**Struktur Notebook:**
1. EDA (Exploratory Data Analysis)
2. Direct Modeling (tanpa preprocessing)
3. Modeling dengan Preprocessing + SMOTE (tanpa data leakage)
4. Hyperparameter Tuning + Feature Selection
5. Deployment (simpan model terbaik)

In [1]:
# ============================================================
# IMPORT SEMUA LIBRARY YANG DIPERLUKAN
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

print('✅ Semua library berhasil diimport!')

ModuleNotFoundError: No module named 'pandas'

---
## BAGIAN 1 – EDA (EXPLORATORY DATA ANALYSIS)

In [ ]:
# ============================================================
# TUGAS 1: 5 BARIS PERTAMA, INFO DATASET, STATISTIK DESKRIPTIF
# ============================================================

# Membaca dataset
nama_file = 'Sales - Marketing customer dataset.csv'
df = pd.read_csv(nama_file)
print(f'✅ Dataset berhasil dimuat! Shape: {df.shape}')

print('\n' + '='*55)
print('   1A. 5 BARIS PERTAMA DATASET')
print('='*55)
display(df.head(5))

print('\n' + '='*55)
print('   1B. INFORMASI DATASET (TIPE DATA & NON-NULL)')
print('='*55)
df.info()

print('\n' + '='*55)
print('   1C. STATISTIK DESKRIPTIF FITUR NUMERIK')
print('='*55)
display(df.describe().T)

In [ ]:
# ============================================================
# TUGAS 2: PERSENTASE MISSING VALUE + VISUALISASI DIAGRAM BATANG
# ============================================================

print('='*55)
print('   2A. PERSENTASE MISSING VALUE PER KOLOM')
print('='*55)

persentase_kosong = (df.isnull().sum() / len(df)) * 100
df_missing = pd.DataFrame({
    'Nama Kolom': persentase_kosong.index,
    'Persentase Kosong (%)': persentase_kosong.values
}).sort_values('Persentase Kosong (%)', ascending=False).reset_index(drop=True)

display(df_missing)

print('\n' + '='*55)
print('   2B. VISUALISASI DIAGRAM BATANG MISSING VALUE')
print('='*55)

# Hanya tampilkan kolom yang memiliki missing value
df_missing_nonzero = df_missing[df_missing['Persentase Kosong (%)'] > 0]

if df_missing_nonzero.empty:
    print('✅ Tidak ada missing value pada dataset ini.')
else:
    plt.figure(figsize=(10, max(4, len(df_missing_nonzero) * 0.4)))
    sns.barplot(
        x='Persentase Kosong (%)', y='Nama Kolom',
        data=df_missing_nonzero, palette='Reds_r'
    )
    plt.title('Persentase Missing Value Per Kolom', fontsize=14, pad=12)
    plt.xlabel('Persentase (%)', fontsize=11)
    plt.ylabel('Kolom', fontsize=11)
    plt.tight_layout()
    plt.show()

# Tetap tampilkan chart penuh semua kolom untuk kelengkapan laporan
plt.figure(figsize=(12, 10))
sns.barplot(
    x='Persentase Kosong (%)', y='Nama Kolom',
    data=df_missing, palette='muted'
)
plt.title('Persentase Missing Value Seluruh Kolom (termasuk 0%)', fontsize=14, pad=12)
plt.xlabel('Persentase Nilai Kosong (%)', fontsize=11)
plt.ylabel('Nama Kolom', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TUGAS 3: DISTRIBUSI VARIABEL TARGET (CHURN)
# ============================================================

print('='*55)
print('   3A. DISTRIBUSI KELAS TARGET (CHURN)')
print('='*55)

churn_counts = df['churn'].value_counts()
churn_pct = df['churn'].value_counts(normalize=True) * 100

print(f'Kelas 0 (Tidak Churn) : {churn_counts[0]:,} data ({churn_pct[0]:.2f}%)')
print(f'Kelas 1 (Churn)       : {churn_counts[1]:,} data ({churn_pct[1]:.2f}%)')
print(f'\n⚠️  Rasio Imbalance   : {churn_pct[0]:.1f}% vs {churn_pct[1]:.1f}%')

if abs(churn_pct[0] - churn_pct[1]) > 10:
    print('   → Dataset IMBALANCED, perlu penanganan (SMOTE akan diterapkan di tahap preprocessing)')
else:
    print('   → Dataset relatif BALANCED')

print('\n' + '='*55)
print('   3B. VISUALISASI DIAGRAM BATANG DISTRIBUSI CHURN')
print('='*55)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart jumlah absolut
sns.countplot(x='churn', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Distribusi Kelas Churn (Jumlah)', fontsize=13)
axes[0].set_xlabel('Churn (0 = Tidak, 1 = Ya)', fontsize=11)
axes[0].set_ylabel('Jumlah Data', fontsize=11)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['0 (Tidak Churn)', '1 (Churn)'])
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart persentase
axes[1].pie(
    churn_counts.values,
    labels=['Tidak Churn (0)', 'Churn (1)'],
    autopct='%1.2f%%',
    colors=['#66c2a5', '#fc8d62'],
    startangle=140,
    explode=(0, 0.07)
)
axes[1].set_title('Proporsi Kelas Target Churn (%)', fontsize=13)

plt.suptitle('Analisis Keseimbangan Kelas Target (Churn)', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TUGAS 4: HEATMAP KORELASI FITUR NUMERIK
# ============================================================

print('='*55)
print('   4. HEATMAP KORELASI FITUR NUMERIK')
print('='*55)

fitur_numerik_eda = df.select_dtypes(include=[np.number]).copy()
if 'customer_id' in fitur_numerik_eda.columns:
    fitur_numerik_eda = fitur_numerik_eda.drop(columns=['customer_id'])

matriks_korelasi = fitur_numerik_eda.corr()

# Tampilkan top 10 fitur berkorelasi dengan churn
korelasi_churn = matriks_korelasi['churn'].drop('churn').abs().sort_values(ascending=False)
print('\nTop 10 Fitur Berkorelasi dengan CHURN:')
print(korelasi_churn.head(10).to_string())

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(matriks_korelasi, dtype=bool))
sns.heatmap(
    matriks_korelasi,
    annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, cbar=True,
    mask=mask,   # Tampilkan setengah segitiga saja agar lebih rapi
    annot_kws={'size': 7}
)
plt.title('Heatmap Korelasi Antar Fitur Numerik (Triangle Bawah)', fontsize=15, pad=20)
plt.tight_layout()
plt.show()

# Heatmap fokus ke churn saja
plt.figure(figsize=(5, 8))
korelasi_churn_df = pd.DataFrame(korelasi_churn).reset_index()
korelasi_churn_df.columns = ['Fitur', 'Korelasi Absolut dengan Churn']
korelasi_churn_df = korelasi_churn_df.head(15)
sns.barplot(x='Korelasi Absolut dengan Churn', y='Fitur', data=korelasi_churn_df, palette='viridis')
plt.title('Top 15 Fitur Berkorelasi dengan Churn', fontsize=13)
plt.tight_layout()
plt.show()

---
## BAGIAN 2 – DIRECT MODELING (Tanpa Preprocessing, Tanpa Tuning)

> Pada skenario ini model dilatih langsung dengan data numerik mentah (hanya fillna median), **tanpa scaling, encoding lengkap, atau SMOTE**.

In [ ]:
# ============================================================
# TUGAS 1 & 2: PENETAPAN X, y DAN TRAIN-TEST SPLIT
# ============================================================

print('='*55)
print('  DIRECT – PENETAPAN X & y')
print('='*55)

# Target
y = df['churn']

# Hanya gunakan fitur numerik, drop customer_id (tidak prediktif)
X_direct = df.drop(columns=['churn']).select_dtypes(include=['int64', 'float64'])
if 'customer_id' in X_direct.columns:
    X_direct = X_direct.drop(columns=['customer_id'])

# Isi missing value dengan median agar model bisa running
X_direct = X_direct.fillna(X_direct.median())

print(f'✅ Fitur X (numerik):  {X_direct.shape}')
print(f'✅ Target y          :  {y.shape}')
print(f'   Kolom fitur       :  {list(X_direct.columns)}')

print('\n' + '='*55)
print('  DIRECT – TRAIN-TEST SPLIT (80:20, stratify=y)')
print('='*55)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_direct, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train : {X_train_d.shape}  |  y_train distribusi: {dict(y_train_d.value_counts())}')
print(f'X_test  : {X_test_d.shape}  |  y_test  distribusi: {dict(y_test_d.value_counts())}')

In [ ]:
# ============================================================
# TUGAS 3 & 4: INISIALISASI & TRAINING 3 KATEGORI MODEL
# ============================================================

print('='*55)
print('  DIRECT – INISIALISASI & TRAINING MODEL')
print('='*55)

# 1. Konvensional: Logistic Regression
model_lr_d = LogisticRegression(max_iter=1000, random_state=42)

# 2. Ensemble Bagging: Random Forest
model_rf_d = RandomForestClassifier(random_state=42)

# 3. Ensemble Voting: LR + KNN + SVM
model_voting_d = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42)),
        ('knn', KNeighborsClassifier()),
        ('svm', SVC(probability=True, random_state=42))
    ],
    voting='soft'
)

print('-> Training Logistic Regression...')
model_lr_d.fit(X_train_d, y_train_d)

print('-> Training Random Forest...')
model_rf_d.fit(X_train_d, y_train_d)

print('-> Training Voting Classifier (tunggu sejenak)...')
model_voting_d.fit(X_train_d, y_train_d)

print('\n✅ Ketiga model (DIRECT) berhasil dilatih!')

In [ ]:
# ============================================================
# TUGAS 5: EVALUASI METRIK + CONFUSION MATRIX
# ============================================================

def evaluasi_model(model, nama_model, X_test, y_test, skenario=''):
    """Fungsi evaluasi universal: cetak metrik + plot confusion matrix."""
    y_pred = model.predict(X_test)

    acc   = accuracy_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, zero_division=0)
    rec   = recall_score(y_test, y_pred, zero_division=0)
    f1    = f1_score(y_test, y_pred, zero_division=0)
    cm    = confusion_matrix(y_test, y_pred)

    label = f'[{skenario}] {nama_model}' if skenario else nama_model
    print(f'\n📊 PERFORMA MODEL: {label.upper()}')
    print(f'   Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
    print(f'   Precision : {prec:.4f} ({prec*100:.2f}%)')
    print(f'   Recall    : {rec:.4f} ({rec*100:.2f}%)')
    print(f'   F1-Score  : {f1:.4f} ({f1*100:.2f}%)')
    print(f'\n   Classification Report:')
    print(classification_report(y_test, y_pred, target_names=['Tidak Churn', 'Churn'], zero_division=0))

    # Plot confusion matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Tidak Churn', 'Churn'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Confusion Matrix\n{label}', fontsize=11)
    plt.tight_layout()
    plt.show()

    return {'model': label, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}


print('='*55)
print('  DIRECT – EVALUASI KETIGA MODEL')
print('='*55)

hasil_direct = []
hasil_direct.append(evaluasi_model(model_lr_d, 'Logistic Regression', X_test_d, y_test_d, 'DIRECT'))
hasil_direct.append(evaluasi_model(model_rf_d, 'Random Forest', X_test_d, y_test_d, 'DIRECT'))
hasil_direct.append(evaluasi_model(model_voting_d, 'Voting Classifier', X_test_d, y_test_d, 'DIRECT'))

---
## BAGIAN 3 – MODELING DENGAN PREPROCESSING + SMOTE

> **Anti Data Leakage:** Semua fitting (imputer median, scaler, SMOTE) dilakukan **hanya pada data training**, kemudian di-transform ke data test. SMOTE **hanya diterapkan pada training set**.

In [ ]:
# ============================================================
# TUGAS 1: PREPROCESSING DATA
# ============================================================

print('='*55)
print('  PREP – STEP 1: FEATURE ENGINEERING & CLEANING')
print('='*55)

df_prep = df.copy()

# --- 1. REKAYASA FITUR TANGGAL ---
df_prep['signup_date']       = pd.to_datetime(df_prep['signup_date'], errors='coerce')
df_prep['last_purchase_date']= pd.to_datetime(df_prep['last_purchase_date'], errors='coerce')
df_prep['tenure_days']       = (df_prep['last_purchase_date'] - df_prep['signup_date']).dt.days
df_prep['tenure_days']       = df_prep['tenure_days'].fillna(0).clip(lower=0)

# --- 2. DROP KOLOM TIDAK RELEVAN ---
# customer_id   → identifier, bukan fitur prediktif
# signup_date & last_purchase_date → sudah dikonversi ke tenure_days
# coupon_code   → high cardinality text, tidak informatif
kolom_drop = ['customer_id', 'signup_date', 'last_purchase_date', 'coupon_code']
df_prep = df_prep.drop(columns=[c for c in kolom_drop if c in df_prep.columns])

print(f'[OK] Kolom setelah drop: {df_prep.shape[1]} fitur + target')

# --- 3. PENANGANAN DUPLIKASI ---
n_dup = df_prep.duplicated().sum()
print(f'[OK] Duplikasi ditemukan: {n_dup} baris' + (' → dihapus' if n_dup > 0 else ' → tidak ada'))
if n_dup > 0:
    df_prep = df_prep.drop_duplicates()

# --- 4. ENCODING KATEGORIKAL (sebelum split agar konsisten) ---
fitur_kat = df_prep.select_dtypes(include='object').columns.tolist()
print(f'[OK] Fitur kategorikal yang di-encode: {fitur_kat}')

# Imputasi modus pada kategorikal SEBELUM encoding (semua data, bukan per train/test)
# Ini boleh karena encoding struktur (kategori mana saja) tidak "bocor" dari test ke train
for col in fitur_kat:
    df_prep[col] = df_prep[col].fillna(df_prep[col].mode()[0])

df_prep = pd.get_dummies(df_prep, columns=fitur_kat, drop_first=True)
df_prep.columns = df_prep.columns.str.replace(' ', '_').str.replace('-', '_')
print(f'[OK] Shape setelah encoding: {df_prep.shape}')

In [ ]:
# ============================================================
# TUGAS 2 & 3: SPLIT DATA (proporsi sama 80:20)
# ============================================================

print('='*55)
print('  PREP – STEP 2: SPLIT DATA (80:20)')
print('='*55)

X_p = df_prep.drop(columns=['churn'])
y_p = df_prep['churn']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_p, y_p, test_size=0.2, random_state=42, stratify=y_p
)

print(f'X_train: {X_train_p.shape}  |  y_train: {dict(y_train_p.value_counts())}')
print(f'X_test : {X_test_p.shape}  |  y_test : {dict(y_test_p.value_counts())}')

# ============================================================
# STEP 3: IMPUTATION NUMERIK (FIT pada TRAIN, TRANSFORM pada TEST)
# ✅ Mencegah data leakage: median dihitung hanya dari training set
# ============================================================

print('\n' + '='*55)
print('  PREP – STEP 3: IMPUTATION (fit ONLY on train)')
print('='*55)

fitur_num = X_train_p.select_dtypes(include=[np.number]).columns.tolist()

# Hitung median HANYA dari data training
median_train = X_train_p[fitur_num].median()

X_train_p = X_train_p.copy()
X_test_p  = X_test_p.copy()

X_train_p[fitur_num] = X_train_p[fitur_num].fillna(median_train)
X_test_p[fitur_num]  = X_test_p[fitur_num].fillna(median_train)   # Gunakan median TRAIN!

print('[OK] Imputation selesai. Missing value tersisa:')
print(f'     Train: {X_train_p.isnull().sum().sum()}  |  Test: {X_test_p.isnull().sum().sum()}')

In [ ]:
# ============================================================
# STEP 4: OUTLIER HANDLING (hanya pada training set)
# ✅ Mencegah data leakage: clip batas IQR dihitung dari train
# ============================================================

print('='*55)
print('  PREP – STEP 4: OUTLIER HANDLING (IQR Capping, train only)')
print('='*55)

# Pilih fitur kontinu (bukan binary 0/1) untuk outlier capping
fitur_kontinu = [
    col for col in fitur_num
    if X_train_p[col].nunique() > 10  # Heuristik: nilai unik > 10 → kontinu
]
print(f'Fitur kontinu yang akan di-cap: {fitur_kontinu}')

# Simpan batas IQR dari training saja
batas_iqr = {}
for col in fitur_kontinu:
    Q1  = X_train_p[col].quantile(0.25)
    Q3  = X_train_p[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    batas_iqr[col] = (lower, upper)
    X_train_p[col] = X_train_p[col].clip(lower=lower, upper=upper)
    X_test_p[col]  = X_test_p[col].clip(lower=lower, upper=upper)  # Gunakan batas TRAIN!

print('[OK] Outlier capping selesai (IQR batas dihitung dari training set)')

In [ ]:
# ============================================================
# STEP 5: SCALING (fit pada TRAIN, transform keduanya)
# ✅ Mencegah data leakage: scaler di-fit hanya dari training set
# ============================================================

print('='*55)
print('  PREP – STEP 5: FEATURE SCALING (fit ONLY on train)')
print('='*55)

scaler = StandardScaler()
X_train_p[fitur_num] = scaler.fit_transform(X_train_p[fitur_num])   # fit + transform train
X_test_p[fitur_num]  = scaler.transform(X_test_p[fitur_num])         # hanya transform test

print('[OK] StandardScaler selesai. Scaler di-fit HANYA dari X_train_p!')
print(f'     Mean (5 fitur pertama): {scaler.mean_[:5].round(3)}')

In [ ]:
# ============================================================
# STEP 6: SMOTE – OVERSAMPLING KELAS MINORITAS
# ✅ SMOTE hanya diterapkan pada data TRAINING, BUKAN pada test!
# ============================================================

print('='*55)
print('  PREP – STEP 6: SMOTE (hanya pada training set!)')
print('='*55)

print(f'Distribusi y_train SEBELUM SMOTE: {dict(y_train_p.value_counts())}')

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_p, y_train_p)

print(f'Distribusi y_train SESUDAH SMOTE  : {dict(pd.Series(y_train_sm).value_counts())}')
print(f'Ukuran X_train setelah SMOTE       : {X_train_sm.shape}')
print('\n✅ SMOTE berhasil menyeimbangkan kelas pada training set!')
print('   (Data test TIDAK tersentuh SMOTE → tidak ada data leakage)')

# Visualisasi distribusi sebelum vs sesudah SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

y_train_p.value_counts().plot(kind='bar', ax=axes[0], color=['#66c2a5', '#fc8d62'], edgecolor='black')
axes[0].set_title('Sebelum SMOTE (Train Set)', fontsize=12)
axes[0].set_xlabel('Kelas Churn'); axes[0].set_ylabel('Jumlah')
axes[0].set_xticklabels(['Tidak Churn (0)', 'Churn (1)'], rotation=0)

pd.Series(y_train_sm).value_counts().plot(kind='bar', ax=axes[1], color=['#66c2a5', '#fc8d62'], edgecolor='black')
axes[1].set_title('Sesudah SMOTE (Train Set)', fontsize=12)
axes[1].set_xlabel('Kelas Churn'); axes[1].set_ylabel('Jumlah')
axes[1].set_xticklabels(['Tidak Churn (0)', 'Churn (1)'], rotation=0)

plt.suptitle('Efek SMOTE pada Distribusi Kelas Target (Training Set)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TUGAS 4 & 5: TRAINING & EVALUASI 3 MODEL DENGAN PREPROCESSING
# ============================================================

print('='*55)
print('  PREP – TRAINING 3 MODEL (data sudah bersih + SMOTE)')
print('='*55)

# 1. Konvensional: Logistic Regression
model_lr_p = LogisticRegression(max_iter=1000, random_state=42)

# 2. Ensemble Bagging: Random Forest
model_rf_p = RandomForestClassifier(random_state=42)

# 3. Ensemble Voting: LR + KNN + SVM
model_voting_p = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42)),
        ('knn', KNeighborsClassifier()),
        ('svm', SVC(probability=True, random_state=42))
    ],
    voting='soft'
)

print('-> Training Logistic Regression...')
model_lr_p.fit(X_train_sm, y_train_sm)

print('-> Training Random Forest...')
model_rf_p.fit(X_train_sm, y_train_sm)

print('-> Training Voting Classifier (tunggu sejenak)...')
model_voting_p.fit(X_train_sm, y_train_sm)

print('\n✅ Ketiga model (PREPROCESSING) berhasil dilatih!')

print('\n' + '='*55)
print('  PREP – EVALUASI KETIGA MODEL')
print('='*55)

hasil_prep = []
hasil_prep.append(evaluasi_model(model_lr_p, 'Logistic Regression', X_test_p, y_test_p, 'PREP'))
hasil_prep.append(evaluasi_model(model_rf_p, 'Random Forest', X_test_p, y_test_p, 'PREP'))
hasil_prep.append(evaluasi_model(model_voting_p, 'Voting Classifier', X_test_p, y_test_p, 'PREP'))

---
## BAGIAN 4 – HYPERPARAMETER TUNING + FEATURE SELECTION

In [ ]:
# ============================================================
# TUGAS 1: ANALISIS FEATURE IMPORTANCE (dari RF yang sudah dilatih)
# ============================================================

print('='*55)
print('  TUNING – STEP 1: FEATURE IMPORTANCE ANALYSIS')
print('='*55)

feature_names = X_train_p.columns.tolist()
importances   = model_rf_p.feature_importances_

df_importance = pd.DataFrame({
    'Fitur'      : feature_names,
    'Importance' : importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('Top 20 Fitur Paling Penting (Random Forest):')      
display(df_importance.head(20))

# Visualisasi
plt.figure(figsize=(9, 8))
sns.barplot(
    x='Importance', y='Fitur',
    data=df_importance.head(20), palette='viridis'
)
plt.title('Top 20 Feature Importance – Random Forest', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

# Pilih top N fitur untuk tuning
TOP_N = 20
top_features = df_importance['Fitur'].head(TOP_N).tolist()
print(f'\n✅ Akan digunakan {TOP_N} fitur teratas untuk hyperparameter tuning:')
print(top_features)

In [ ]:
# ============================================================
# PERSIAPAN DATA TUNING DENGAN FITUR TERPILIH + SMOTE ULANG
# ============================================================

X_train_top = pd.DataFrame(X_train_sm, columns=feature_names)[top_features]
X_test_top  = X_test_p[top_features]

print(f'Shape X_train (top features + SMOTE): {X_train_top.shape}')
print(f'Shape X_test  (top features)        : {X_test_top.shape}')

In [ ]:
# ============================================================
# TUGAS 2 & 3: HYPERPARAMETER TUNING – LOGISTIC REGRESSION
# ============================================================

print('='*55)
print('  TUNING – LOGISTIC REGRESSION (RandomizedSearchCV)')
print('='*55)

param_lr = {
    'C'       : [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty' : ['l1', 'l2'],
    'solver'  : ['liblinear', 'saga'],
    'max_iter': [500, 1000]
}

rs_lr = RandomizedSearchCV(
    LogisticRegression(random_state=42),
    param_distributions=param_lr,
    n_iter=20, cv=5, scoring='f1',
    n_jobs=-1, random_state=42, verbose=0
)

print('-> Mencari parameter terbaik untuk Logistic Regression...')
rs_lr.fit(X_train_top, y_train_sm)
best_lr = rs_lr.best_estimator_

print(f'\n✅ Best Params LR  : {rs_lr.best_params_}')
print(f'   Best CV F1-Score: {rs_lr.best_score_:.4f}')

hasil_tuning_lr = evaluasi_model(best_lr, 'Logistic Regression (Tuned)', X_test_top, y_test_p, 'TUNED')

In [ ]:
# ============================================================
# TUGAS 2 & 3: HYPERPARAMETER TUNING – RANDOM FOREST
# ============================================================

print('='*55)
print('  TUNING – RANDOM FOREST (GridSearchCV)')
print('='*55)

param_rf = {
    'n_estimators'     : [50, 100, 150],
    'max_depth'        : [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4]
}

gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_rf,
    cv=5, scoring='f1',
    n_jobs=-1, verbose=1
)

print('-> Mencari parameter terbaik untuk Random Forest (tunggu)...')
gs_rf.fit(X_train_top, y_train_sm)
best_rf = gs_rf.best_estimator_

print(f'\n✅ Best Params RF   : {gs_rf.best_params_}')
print(f'   Best CV F1-Score  : {gs_rf.best_score_:.4f}')

hasil_tuning_rf = evaluasi_model(best_rf, 'Random Forest (Tuned)', X_test_top, y_test_p, 'TUNED')

In [ ]:
# ============================================================
# TUGAS 2 & 3: HYPERPARAMETER TUNING – VOTING CLASSIFIER
# ============================================================

print('='*55)
print('  TUNING – VOTING CLASSIFIER (RandomizedSearchCV)')
print('='*55)

param_voting = {
    'lr__C'        : [0.1, 1, 10],
    'knn__n_neighbors': [3, 5, 7],
    'knn__weights' : ['uniform', 'distance'],
    'svm__C'       : [0.1, 1, 10],
    'svm__kernel'  : ['rbf', 'linear']
}

voting_base = VotingClassifier(
    estimators=[
        ('lr' , LogisticRegression(max_iter=1000, random_state=42)),
        ('knn', KNeighborsClassifier()),
        ('svm', SVC(probability=True, random_state=42))
    ],
    voting='soft'
)

rs_voting = RandomizedSearchCV(
    voting_base,
    param_distributions=param_voting,
    n_iter=10, cv=5, scoring='f1',
    n_jobs=-1, random_state=42, verbose=0
)

print('-> Mencari parameter terbaik untuk Voting Classifier (tunggu)...')
rs_voting.fit(X_train_top, y_train_sm)
best_voting = rs_voting.best_estimator_

print(f'\n✅ Best Params Voting: {rs_voting.best_params_}')
print(f'   Best CV F1-Score   : {rs_voting.best_score_:.4f}')

hasil_tuning_voting = evaluasi_model(best_voting, 'Voting Classifier (Tuned)', X_test_top, y_test_p, 'TUNED')

In [ ]:
# ============================================================
# PERBANDINGAN SELURUH 9 MODEL (3 skenario × 3 model)
# ============================================================

print('='*60)
print('   REKAP PERBANDINGAN PERFORMA SELURUH 9 MODEL')
print('='*60)

semua_hasil = hasil_direct + hasil_prep + [
    hasil_tuning_lr, hasil_tuning_rf, hasil_tuning_voting
]

df_rekap = pd.DataFrame(semua_hasil)
df_rekap = df_rekap.sort_values('f1', ascending=False).reset_index(drop=True)
df_rekap.index += 1

display(df_rekap.style.format({
    'accuracy': '{:.4f}', 'precision': '{:.4f}',
    'recall'  : '{:.4f}', 'f1'       : '{:.4f}'
}).background_gradient(subset=['f1'], cmap='Greens'))

# Visualisasi perbandingan
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrik_list = ['accuracy', 'precision', 'recall', 'f1']

for ax, metrik in zip(axes.flatten(), metrik_list):
    df_plot = df_rekap.sort_values(metrik, ascending=False)
    bars = ax.barh(df_plot['model'], df_plot[metrik],
                   color=plt.cm.viridis(np.linspace(0.2, 0.85, len(df_plot))))
    ax.set_title(metrik.upper(), fontsize=12)
    ax.set_xlabel('Score')
    ax.set_xlim(0, 1)
    ax.tick_params(axis='y', labelsize=7)
    for bar, val in zip(bars, df_plot[metrik]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

plt.suptitle('Perbandingan Performa 9 Model – Semua Skenario', fontsize=14, y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

# Identifikasi model terbaik
best_row = df_rekap.iloc[0]
print(f'\n🏆 MODEL TERBAIK: {best_row["model"]}')
print(f'   F1-Score : {best_row["f1"]:.4f}')
print(f'   Accuracy : {best_row["accuracy"]:.4f}')

---
## BAGIAN 5 – DEPLOYMENT

Simpan model terbaik beserta scaler dan daftar fitur agar bisa dimuat di aplikasi Streamlit.

In [ ]:
# ============================================================
# TUGAS 1 & 2: SIMPAN MODEL TERBAIK + ARTEFAK PREPROCESSING
# ============================================================

print('='*55)
print('  DEPLOYMENT – SIMPAN ARTEFAK MODEL')
print('='*55)

os.makedirs('deployment', exist_ok=True)

# Tentukan model terbaik secara otomatis dari rekap
nama_model_terbaik = best_row['model']
print(f'-> Model terbaik yang akan disimpan: {nama_model_terbaik}')

# Peta nama → objek model
peta_model = {
    '[DIRECT] Logistic Regression'         : model_lr_d,
    '[DIRECT] Random Forest'               : model_rf_d,
    '[DIRECT] Voting Classifier'           : model_voting_d,
    '[PREP] Logistic Regression'           : model_lr_p,
    '[PREP] Random Forest'                 : model_rf_p,
    '[PREP] Voting Classifier'             : model_voting_p,
    '[TUNED] Logistic Regression (Tuned)'  : best_lr,
    '[TUNED] Random Forest (Tuned)'        : best_rf,
    '[TUNED] Voting Classifier (Tuned)'    : best_voting,
}

model_terbaik = peta_model.get(nama_model_terbaik, best_rf)

# Simpan model terbaik
joblib.dump(model_terbaik, 'deployment/best_model.pkl')

# Simpan scaler
joblib.dump(scaler, 'deployment/scaler.pkl')

# Simpan daftar fitur yang digunakan (top_features)
joblib.dump(top_features, 'deployment/feature_names.pkl')

# Simpan batas IQR untuk outlier handling di inference
joblib.dump(batas_iqr, 'deployment/iqr_bounds.pkl')

# Simpan median training untuk imputation di inference
joblib.dump(median_train, 'deployment/median_train.pkl')

print('\n✅ Artefak deployment berhasil disimpan:')
for f in os.listdir('deployment'):
    size = os.path.getsize(f'deployment/{f}') / 1024
    print(f'   deployment/{f}  ({size:.1f} KB)')

In [ ]:
# ============================================================
# VERIFIKASI: Test load model & prediksi sample
# ============================================================

print('='*55)
print('  DEPLOYMENT – VERIFIKASI LOAD & PREDIKSI')
print('='*55)

loaded_model    = joblib.load('deployment/best_model.pkl')
loaded_scaler   = joblib.load('deployment/scaler.pkl')
loaded_features = joblib.load('deployment/feature_names.pkl')

# Ambil 5 sampel dari test set sebagai simulasi input baru
sample_raw  = X_test_top.iloc[:5].copy()
sample_pred = loaded_model.predict(sample_raw)
sample_prob = loaded_model.predict_proba(sample_raw)[:, 1] if hasattr(loaded_model, 'predict_proba') else None

print('\nHasil prediksi 5 sampel pertama dari test set:')
df_result = pd.DataFrame({
    'Prediksi Churn'   : sample_pred,
    'Label Aktual'     : y_test_p.iloc[:5].values,
    'Probabilitas Churn': sample_prob if sample_prob is not None else 'N/A'
})
df_result['Prediksi'] = df_result['Prediksi Churn'].map({0: 'Tidak Churn', 1: 'Churn'})
df_result['Aktual']   = df_result['Label Aktual'].map({0: 'Tidak Churn', 1: 'Churn'})
display(df_result)

print('\n✅ Model berhasil dimuat dan berjalan normal. Siap untuk deployment Streamlit!')

In [ ]:
# ============================================================
# GENERATE: app.py untuk Streamlit
# ============================================================

streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# ── Load Artefak ────────────────────────────────────────────
@st.cache_resource
def load_artifacts():
    model        = joblib.load("deployment/best_model.pkl")
    scaler       = joblib.load("deployment/scaler.pkl")
    features     = joblib.load("deployment/feature_names.pkl")
    median_train = joblib.load("deployment/median_train.pkl")
    iqr_bounds   = joblib.load("deployment/iqr_bounds.pkl")
    return model, scaler, features, median_train, iqr_bounds

model, scaler, FEATURES, MEDIAN_TRAIN, IQR_BOUNDS = load_artifacts()

# ── UI ──────────────────────────────────────────────────────
st.set_page_config(page_title="Prediksi Churn Pelanggan", page_icon="📊", layout="wide")

st.title("📊 Prediksi Churn Pelanggan")
st.markdown("**UAS Bengkel Koding Data Science** – Model Machine Learning untuk Memprediksi Customer Churn")
st.divider()

st.sidebar.header("Input Fitur Pelanggan")
st.sidebar.markdown("Isi form berikut sesuai data pelanggan:")

# ── Input Form ──────────────────────────────────────────────
with st.sidebar:
    age                    = st.number_input("Usia (age)", 18, 100, 35)
    total_visits           = st.number_input("Total Kunjungan", 0, 1000, 50)
    avg_session_time       = st.number_input("Avg Session Time (menit)", 0.0, 300.0, 15.0)
    pages_per_session      = st.number_input("Pages per Session", 0.0, 50.0, 5.0)
    email_open_rate        = st.slider("Email Open Rate", 0.0, 1.0, 0.3)
    email_click_rate       = st.slider("Email Click Rate", 0.0, 1.0, 0.1)
    total_spent            = st.number_input("Total Spent (Rp)", 0.0, 1e7, 500000.0)
    avg_order_value        = st.number_input("Avg Order Value (Rp)", 0.0, 1e6, 100000.0)
    support_tickets        = st.number_input("Jumlah Support Tickets", 0, 50, 2)
    satisfaction_score     = st.slider("Satisfaction Score", 0.0, 10.0, 7.0)
    nps_score              = st.slider("NPS Score", -100, 100, 30)
    lifetime_value         = st.number_input("Lifetime Value (Rp)", 0.0, 1e8, 2000000.0)
    last_3_month_purchase_freq = st.number_input("Frekuensi Pembelian 3 Bulan", 0, 100, 5)
    tenure_days            = st.number_input("Tenure Days", 0, 5000, 365)
    is_premium_user        = st.selectbox("Premium User?", [0, 1], format_func=lambda x: "Ya" if x else "Tidak")
    discount_used          = st.selectbox("Diskon Digunakan?", [0, 1], format_func=lambda x: "Ya" if x else "Tidak")
    refund_requested       = st.selectbox("Refund Diminta?", [0, 1], format_func=lambda x: "Ya" if x else "Tidak")
    delivery_delay_days    = st.number_input("Delay Pengiriman (hari)", 0, 30, 2)
    marketing_spend_per_user = st.number_input("Marketing Spend per User", 0.0, 1e5, 5000.0)

# ── Prediksi ────────────────────────────────────────────────
if st.button("🔍 Prediksi Churn", use_container_width=True, type="primary"):
    # Buat DataFrame dari input
    input_dict = {
        "age": age, "total_visits": total_visits,
        "avg_session_time": avg_session_time, "pages_per_session": pages_per_session,
        "email_open_rate": email_open_rate, "email_click_rate": email_click_rate,
        "total_spent": total_spent, "avg_order_value": avg_order_value,
        "support_tickets": support_tickets, "satisfaction_score": satisfaction_score,
        "nps_score": nps_score, "lifetime_value": lifetime_value,
        "last_3_month_purchase_freq": last_3_month_purchase_freq,
        "tenure_days": tenure_days, "is_premium_user": is_premium_user,
        "discount_used": discount_used, "refund_requested": refund_requested,
        "delivery_delay_days": delivery_delay_days,
        "marketing_spend_per_user": marketing_spend_per_user
    }

    input_df = pd.DataFrame([input_dict])

    # Tambahkan kolom yang ada di FEATURES tapi tidak di input (isi dengan 0)
    for col in FEATURES:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[FEATURES]

    # Imputation menggunakan median training
    for col in input_df.columns:
        if col in MEDIAN_TRAIN.index:
            input_df[col] = input_df[col].fillna(MEDIAN_TRAIN[col])

    # Prediksi
    prediksi  = model.predict(input_df)[0]
    probas    = model.predict_proba(input_df)[0] if hasattr(model, "predict_proba") else None

    # Tampilkan hasil
    st.subheader("📋 Hasil Prediksi")
    col1, col2 = st.columns(2)

    with col1:
        if prediksi == 1:
            st.error("⚠️ Pelanggan BERPOTENSI CHURN")
        else:
            st.success("✅ Pelanggan TIDAK BERPOTENSI CHURN")

    with col2:
        if probas is not None:
            prob_churn = probas[1]
            st.metric("Probabilitas Churn", f"{prob_churn:.2%}")
            st.progress(float(prob_churn))

    st.divider()
    st.subheader("📌 Rekomendasi")
    if prediksi == 1:
        st.warning("""
        **Tindakan yang disarankan:**
        - Hubungi pelanggan dengan penawaran retensi khusus
        - Berikan diskon atau kupon eksklusif
        - Tingkatkan layanan customer support
        - Lakukan survei kepuasan untuk memahami keluhan
        """)
    else:
        st.info("""
        **Pelanggan ini loyal. Pertahankan dengan:**
        - Program loyalitas & reward
        - Personalisasi komunikasi marketing
        - Penawaran upgrade ke premium (jika belum)
        """)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code)

print('✅ app.py Streamlit berhasil dibuat!')

# Generate requirements.txt
reqs = '''streamlit>=1.28.0
pandas>=1.5.0
numpy>=1.23.0
scikit-learn>=1.3.0
imbalanced-learn>=0.11.0
matplotlib>=3.6.0
seaborn>=0.12.0
joblib>=1.2.0
'''

with open('requirements.txt', 'w') as f:
    f.write(reqs)

print('✅ requirements.txt berhasil dibuat!')
print('\n📁 File siap untuk deployment ke Streamlit Cloud:')
print('   - app.py')
print('   - requirements.txt')
print('   - deployment/best_model.pkl')
print('   - deployment/scaler.pkl')
print('   - deployment/feature_names.pkl')
print('   - deployment/median_train.pkl')
print('   - deployment/iqr_bounds.pkl')

---
## RINGKASAN PROYEK

| Skenario | Model | Catatan |
|---|---|---|
| Direct | Logistic Regression | Fitur numerik saja, fillna median |
| Direct | Random Forest | Fitur numerik saja, fillna median |
| Direct | Voting (LR+KNN+SVM) | Fitur numerik saja, fillna median |
| Preprocessing | Logistic Regression | Full prep + SMOTE |
| Preprocessing | Random Forest | Full prep + SMOTE |
| Preprocessing | Voting (LR+KNN+SVM) | Full prep + SMOTE |
| Tuning | LR (RandomizedSearchCV) | Top 20 features + SMOTE |
| Tuning | RF (GridSearchCV) | Top 20 features + SMOTE |
| Tuning | Voting (RandomizedSearchCV) | Top 20 features + SMOTE |

**Anti Data Leakage Checklist:**
- ✅ Split dilakukan SEBELUM imputation, scaling, dan SMOTE
- ✅ Median imputation dihitung HANYA dari training set
- ✅ IQR bounds dihitung HANYA dari training set
- ✅ StandardScaler di-fit HANYA dari training set
- ✅ SMOTE diterapkan HANYA pada training set
- ✅ Encoding (get_dummies) dilakukan sebelum split (tidak bocor nilai, hanya struktur kolom)